View all columns in a table

In [0]:
SELECT *
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified
LIMIT 15

View if features about international students represent same information

In [0]:
SELECT DISTINCT a.is_international_student, a.international_domestic_student
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified AS a
JOIN workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified AS b
ON a.is_commencing_period = b.is_commencing_period AND a.commencing_continuing_period = b.commencing_continuing_period

View count per group

In [0]:
SELECT course_level, course_group, COUNT(*)
FROM workspace.student_aggregate.dwh_curriculum__course AS course
GROUP BY course_level, course_group
ORDER BY course_level, course_group

Checking commencing years

In [0]:
SELECT enrolment_year AS commencing_year, student_deidentified_hash
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified AS stud
WHERE is_commencing = TRUE
LIMIT 20

Trying to identify continuing students' commencing years (failed because each row is a unique student)

In [0]:
WITH commencing_years AS (
    SELECT enrolment_year AS commencing_year, student_deidentified_hash
    FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified AS stud
    WHERE is_commencing = TRUE
)
SELECT attendance_mode, commencing_year, is_twelve_month_course_attrition, COUNT(*)
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified AS stud_data
INNER JOIN commencing_years
    ON stud_data.student_deidentified_hash = commencing_years.student_deidentified_hash
GROUP BY attendance_mode, commencing_year, is_twelve_month_course_attrition
ORDER BY attendance_mode, commencing_year, is_twelve_month_course_attrition

Joining student data with Study Area A data

In [0]:
SELECT narrow_field_of_education, study_a.study_area_a_key, COUNT(*)
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified AS stud_data
LEFT JOIN workspace.student_aggregate.dwh_curriculum__study_area_a AS study_a
ON stud_data.study_area_a_key_hash = study_a.study_area_a_key_hash
GROUP BY narrow_field_of_education, study_a.study_area_a_key
ORDER BY narrow_field_of_education, study_a.study_area_a_key

Check combinations of narrow and detailed fields of education, along with commencing/continuing status

In [0]:
SELECT narrow_primary_field_of_education, detailed_primary_field_of_education, commencing_continuing, COUNT(*)
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified AS s
LEFT JOIN workspace.student_aggregate.dwh_curriculum__course AS c
ON s.course_key_hash = c.course_key_hash
WHERE broad_primary_field_of_education LIKE "HEALTH"
GROUP BY narrow_primary_field_of_education, detailed_primary_field_of_education, commencing_continuing
ORDER BY narrow_primary_field_of_education, detailed_primary_field_of_education, commencing_continuing

Check combinations of commencing/continuing variants to see which ones always have the same information

In [0]:
SELECT DISTINCT commencing_continuing, commencing_continuing_12m, commencing_continuing_half_year, commencing_continuing_period
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified

Check is_commencing variants to see if they have the same combinations as commencing/continuing variants and hence, are able to be removed as they are redundant

In [0]:
SELECT DISTINCT is_commencing, is_commencing_12m, is_commencing_half_year, is_commencing_period
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified

Check that each student does, in fact, only show one record in the synthetic data

In [0]:
SELECT student_deidentified_hash, COUNT(*) AS num_enrolments
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified
GROUP BY student_deidentified_hash
ORDER BY num_enrolments DESC

View the range of EFTSL present in the synthetic data

In [0]:
SELECT DISTINCT eftsl
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified
ORDER BY eftsl DESC

Validation - Attrition Rate by Categories

In [0]:
SELECT cumulative_credit_points_withdrawn, is_twelve_month_course_attrition, COUNT(*)
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified AS stud
LEFT JOIN workspace.student_aggregate.dwh_curriculum__course AS c
ON stud.course_key_hash = c.course_key_hash
GROUP BY cumulative_credit_points_withdrawn, is_twelve_month_course_attrition
ORDER BY cumulative_credit_points_withdrawn, is_twelve_month_course_attrition

Validation - Reproduce distributions provided by industry partners

In [0]:
CREATE OR REPLACE TEMP VIEW my_temp_view AS
SELECT broad_primary_field_of_education AS field_of_education,
CASE
    WHEN cumulative_credit_points_withdrawn < 24 THEN '0-24'
    WHEN cumulative_credit_points_withdrawn < 48 THEN '24-48'
    WHEN cumulative_credit_points_withdrawn < 96 THEN '48-96'
    WHEN cumulative_credit_points_withdrawn < 144 THEN '96-144'
    ELSE '144+'
END AS credit_point_band,
COUNT(*) AS student_count
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified AS s
LEFT JOIN workspace.student_aggregate.dwh_curriculum__course AS c
ON s.course_key_hash = c.course_key_hash
GROUP BY field_of_education, credit_point_band
ORDER BY field_of_education, credit_point_band;

SELECT * FROM my_temp_view;

In [0]:
CREATE OR REPLACE TEMP VIEW my_temp_view AS
SELECT broad_primary_field_of_education AS field_of_education,
teaching_period,
CASE
    WHEN cumulative_credit_points_withdrawn < 0.125 THEN '0-0.125'
    WHEN cumulative_credit_points_withdrawn < 0.25 THEN '0.125-0.25'
    WHEN cumulative_credit_points_withdrawn < 0.5 THEN '0.25-0.5'
    WHEN cumulative_credit_points_withdrawn < 0.75 THEN '0.5-0.75'
    WHEN cumulative_credit_points_withdrawn < 1 THEN '0.75-1.0'
    ELSE '1.0+'
END AS eftsl_band,
COUNT(*) AS student_count
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified AS s
LEFT JOIN workspace.student_aggregate.dwh_curriculum__course AS c
ON s.course_key_hash = c.course_key_hash
LEFT JOIN workspace.student_aggregate.dwh_learning_and_teaching__teaching_period AS t
ON s.teaching_period_key = t.teaching_period_key
GROUP BY field_of_education, teaching_period, eftsl_band
ORDER BY field_of_education, teaching_period, eftsl_band;

SELECT * FROM my_temp_view;

In [0]:
SELECT course_admission_load_category, COUNT(*)
FROM workspace.student_aggregate.rpt_student_management__fact__all_enrolment_eftsl__deidentified
GROUP BY course_admission_load_category
ORDER BY course_admission_load_category